# Deep Learning Forecasting (Optional)

This notebook compares three small neural forecasting models: an MLP, a GRU and an LSTM. Each learns one-step sequences from the training history and is evaluated with recursive 28-day forecasts. The development folds end before the final 28 calendar dates, which are reserved for a single post-selection assessment.

The sample is small, so architectures and training budgets are deliberately constrained. All models are compared with naive and weekly seasonal-naive forecasts on the same eligible folds.


In [1]:
# PROJECT_ROOT_BOOTSTRAP: support kernels started from the repository root or notebooks/.
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Could not locate the project root containing src/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from importlib.metadata import version
from importlib.util import find_spec
import json
from pathlib import Path
import numpy as np
import pandas as pd
if find_spec("tensorflow") is None:
    print("TensorFlow is not installed. Install requirements-deep-learning.txt to run this optional notebook.")
else:
    import tensorflow as tf
    from src.data import load_demand, TARGETS
    from src.backtesting import naive_forecast, seasonal_naive_forecast
    from src.evaluation import expanding_splits, forecast_metrics
    from src.models.deep_learning_models import fit_sequence_model, recursive_forecast, SUPPORTED_ARCHITECTURES

    SEED = 42
    LOOKBACK = 28
    HORIZON = 28
    MAX_EPOCHS = 30
    PATIENCE = 4

    tf.keras.utils.set_random_seed(SEED)
    try:
        tf.config.experimental.enable_op_determinism()
    except (AttributeError, RuntimeError):
        pass

    demand = load_demand(ROOT / "data" / "raw" / "clientes_restaurantes.csv")
    demand["Total demand"] = demand[list(TARGETS)].sum(axis=1, min_count=2)
    development = demand.iloc[:-HORIZON].copy()
    final_holdout = demand.iloc[-HORIZON:].copy()
    fold_rows = []
    epoch_rows = []
    prediction_rows = []

    for target_number, target in enumerate([*TARGETS, "Total demand"]):
        series = development[target]
        splits = list(expanding_splits(len(series), initial=280, horizon=HORIZON, step=HORIZON))
        for fold_number, (train_slice, test_slice) in enumerate(splits, start=1):
            train = series.iloc[train_slice].copy()
            test = series.iloc[test_slice].copy()
            if test.isna().any():
                print(target, fold_number, "excluded from scoring; unknown test dates:", test.index[test.isna()].tolist())
                continue
            missing_train = train.index[train.isna()]
            if len(missing_train):
                print(target, fold_number, "training-only interpolation dates:", missing_train.tolist())
            train = train.interpolate(limit_area="inside").dropna()
            baseline_predictions = {
                "naive": naive_forecast(train, test.index),
                "seasonal_naive_7": seasonal_naive_forecast(train, test.index, period=7),
            }
            for model_name, prediction in baseline_predictions.items():
                fold_rows.append({"restaurant": target, "fold": fold_number, "model": model_name,
                                  **forecast_metrics(test, prediction, train)})
                for step, (date, value) in enumerate(prediction.items(), start=1):
                    prediction_rows.append({"date": date, "restaurant": target, "model": model_name,
                                            "fold": fold_number, "horizon": step, "prediction": value,
                                            "lower": np.nan, "upper": np.nan})
            for architecture_number, architecture in enumerate(SUPPORTED_ARCHITECTURES):
                seed = SEED + target_number * 1000 + fold_number * 10 + architecture_number
                model, centre, scale, best_epoch = fit_sequence_model(
                    train.to_numpy(), architecture, lookback=LOOKBACK, seed=seed,
                    max_epochs=MAX_EPOCHS, patience=PATIENCE, validation_fraction=0.2,
                )
                prediction = recursive_forecast(
                    model, train.to_numpy(), len(test), lookback=LOOKBACK,
                    centre=centre, scale=scale, index=test.index,
                )
                model_name = f"tensorflow_{architecture}"
                fold_rows.append({"restaurant": target, "fold": fold_number, "model": model_name,
                                  **forecast_metrics(test, prediction, train)})
                epoch_rows.append({"restaurant": target, "fold": fold_number,
                                   "architecture": architecture, "best_epoch": best_epoch,
                                   "training_cutoff": str(train.index.max().date())})
                for step, (date, value) in enumerate(prediction.items(), start=1):
                    prediction_rows.append({"date": date, "restaurant": target, "model": model_name,
                                            "fold": fold_number, "horizon": step, "prediction": value,
                                            "lower": np.nan, "upper": np.nan})

    fold_results = pd.DataFrame(fold_rows)
    epoch_results = pd.DataFrame(epoch_rows)
    summary = fold_results.groupby(["restaurant", "model"]).agg(
        mae_mean=("mae", "mean"), mae_std=("mae", "std"),
        rmse_mean=("rmse", "mean"), wape_mean=("wape", "mean"),
        smape_mean=("smape", "mean"), mase_mean=("mase", "mean"),
        mean_error=("mean_error", "mean"), folds=("fold", "nunique"),
    ).reset_index()
    display(summary.sort_values(["restaurant", "mae_mean"]))

    Path("reports").mkdir(exist_ok=True)
    Path("artifacts/forecasts").mkdir(parents=True, exist_ok=True)
    Path("artifacts/models").mkdir(parents=True, exist_ok=True)
    fold_results.to_csv("reports/deep_learning_fold_comparison.csv", index=False)
    summary.to_csv("reports/deep_learning_model_comparison.csv", index=False)
    epoch_results.to_csv("reports/deep_learning_training_epochs.csv", index=False)

    # Select one neural architecture per series using development-fold MAE only, then refit on all development data.
    selected = {}
    final_predictions = []
    for target in [*TARGETS, "Total demand"]:
        neural = summary[(summary["restaurant"] == target) & summary["model"].str.startswith("tensorflow_")]
        best = neural.sort_values("mae_mean").iloc[0]
        architecture = str(best["model"]).replace("tensorflow_", "")
        selected_epochs = int(round(epoch_results.loc[
            (epoch_results["restaurant"] == target) & (epoch_results["architecture"] == architecture),
            "best_epoch",
        ].median()))
        dev_series = development[target].interpolate(limit_area="inside").dropna()
        final_model, centre, scale, _ = fit_sequence_model(
            dev_series.to_numpy(), architecture, lookback=LOOKBACK, seed=SEED + len(selected) * 100,
            fixed_epochs=max(1, selected_epochs), validation_fraction=0.0,
        )
        # The model is refitted before the holdout; no holdout targets enter training or selection.
        forecast = recursive_forecast(
            final_model, dev_series.to_numpy(), HORIZON, lookback=LOOKBACK,
            centre=centre, scale=scale, index=final_holdout.index,
        )
        safe_target = target.replace(" & ", "_and_").replace(" ", "_")
        model_path = Path("artifacts/models") / f"tensorflow_{safe_target}.keras"
        final_model.save(model_path)
        selected[target] = {
            "architecture": architecture,
            "development_mean_mae": float(best["mae_mean"]),
            "refit_epochs": max(1, selected_epochs),
            "training_cutoff": str(dev_series.index.max().date()),
            "model_path": str(model_path),
        }
        for step, (date, value) in enumerate(forecast.items(), start=1):
            final_predictions.append({"date": date, "restaurant": target,
                                      "model": f"tensorflow_{architecture}", "horizon": step,
                                      "prediction": value, "lower": np.nan, "upper": np.nan})

    pd.DataFrame(prediction_rows).to_csv("artifacts/forecasts/deep_learning_walk_forward.csv", index=False)
    pd.DataFrame(final_predictions).to_csv("artifacts/forecasts/deep_learning_holdout_forecasts.csv", index=False)
    metadata = {
        "validation_method": "aligned recursive expanding-window development CV; final 28 calendar dates excluded from selection",
        "training_cutoff": str(development.index.max().date()),
        "final_holdout_period": [str(final_holdout.index.min().date()), str(final_holdout.index.max().date())],
        "forecast_horizon_days": HORIZON,
        "parameters": {"lookback": LOOKBACK, "max_epochs": MAX_EPOCHS, "early_stopping_patience": PATIENCE,
                       "validation_fraction": 0.2, "seed": SEED, "architectures": list(SUPPORTED_ARCHITECTURES)},
        "package_versions": {"tensorflow": version("tensorflow"), "numpy": version("numpy"), "pandas": version("pandas")},
        "selected_models": selected,
        "uncertainty_intervals": "not estimated for neural forecasts",
    }
    Path("artifacts/models/deep_learning_metadata.json").write_text(json.dumps(metadata, indent=2))
    print("Neural architectures selected from development folds:", selected)


I0000 00:00:1794691869.088267  220148 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794691869.168689  220148 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1794691871.134559  220148 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Chimi & Churri 1 training-only interpolation dates: [Timestamp('2016-04-05 00:00:00'), Timestamp('2016-09-17 00:00:00')]


I0000 00:00:1794691872.264200  220148 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1766 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


E0000 00:00:1794691873.244899  220148 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


I0000 00:00:1794691878.598687  220240 cuda_dnn.cc:461] Loaded cuDNN version 91500


... (53 outputs left out)

,restaurant,model,mae_mean,mae_std,rmse_mean,wape_mean,smape_mean,mase_mean,mean_error,folds
4,Assa Frão,tensorflow_mlp,15.663801,1.625237,21.672225,0.459058,0.461560,0.817905,0.644129,5
3,Assa Frão,tensorflow_lstm,18.528101,2.295615,24.516220,0.543663,0.545652,0.968098,-1.847702,5
2,Assa Frão,tensorflow_gru,18.717294,2.306979,23.965517,0.549533,0.551144,0.978145,0.741663,5
1,Assa Frão,seasonal_naive_7,18.750000,3.869029,26.641453,0.548611,0.504755,0.979335,2.292857,5
0,Assa Frão,naive,19.464286,3.493069,26.726687,0.568289,0.587756,1.016483,-8.192857,5
7,Chimi & Churri,tensorflow_gru,13.795024,2.853912,17.470772,0.352125,0.392362,0.890046,-3.566169,6
9,Chimi & Churri,tensorflow_mlp,13.967435,1.400107,17.642592,0.356817,0.398487,0.901174,-3.335431,6
8,Chimi & Churri,tensorflow_lstm,15.245576,2.795781,19.824354,0.389928,0.418446,0.983981,-3.765311,6
6,Chimi & Churri,seasonal_naive_7,16.851190,4.331742,20.607658,0.436107,0.480394,1.087007,0.922619,6
5,Chimi & Churri,naive,22.208333,6.575442,28.216037,0.572279,0.622572,1.432332,-5.458333,6


E0000 00:00:1794692160.267024  220148 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


E0000 00:00:1794692167.536300  220148 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


Neural architectures selected from development folds: {'Chimi & Churri': {'architecture': 'gru', 'development_mean_mae': 13.795023590340007, 'refit_epochs': 26, 'training_cutoff': '2017-03-25', 'model_path': 'artifacts/models/tensorflow_Chimi_and_Churri.keras'}, 'Assa Frão': {'architecture': 'mlp', 'development_mean_mae': 15.663801395056606, 'refit_epochs': 8, 'training_cutoff': '2017-03-25', 'model_path': 'artifacts/models/tensorflow_Assa_Frão.keras'}, 'Total demand': {'architecture': 'mlp', 'development_mean_mae': 21.472245790070744, 'refit_epochs': 16, 'training_cutoff': '2017-03-25', 'model_path': 'artifacts/models/tensorflow_Total_demand.keras'}}
